# CSET343 – AI in Healthcare
## Lab Assignment 6: CNN-Based Chest X-Ray Classification

**Objective:** Design and evaluate CNN models for automated diagnosis/classification using chest X-ray images.

> **Assignment dataset:** Chest X-Ray Images with two classes: **NORMAL** and **PNEUMONIA**.  
> The supplied assignment specifies approximately 5,863 images and asks for HOG, LBP, GLCM feature extraction, a 70/15/15 split, CNN training, cross-validation/hyperparameter tuning, evaluation metrics, and a confusion matrix.

**Note:** The assignment's objective mentions heart disease, while the specified dataset/classes are for pneumonia. This notebook follows the **dataset and tasks explicitly listed in the assignment**. fileciteturn0file0L7-L15


## 1. Install and import required libraries

The notebook uses TensorFlow/Keras for the CNN and scikit-image/scikit-learn for HOG, LBP, GLCM, splitting, metrics, and cross-validation.


In [ ]:
!pip -q install scikit-image kagglehub

import os
import random
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter

from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (128, 128)
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Task 1 – Data Acquisition

The standard Kaggle **Chest X-Ray Images (Pneumonia)** dataset is used. It contains `train`, `test`, and `val` folders with `NORMAL` and `PNEUMONIA` classes.

### Kaggle setup
1. Download your `kaggle.json` from Kaggle → Account → Create New API Token.
2. Run the next cell.
3. Upload `kaggle.json` when prompted.


In [ ]:
from google.colab import files

uploaded = files.upload()  # Upload kaggle.json

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("Please upload a file named kaggle.json")

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API configured.")


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Download the dataset
!mkdir -p /content/chest_xray
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/chest_xray --unzip

DATA_ROOT = Path("/content/chest_xray/chest_xray")
print("Dataset root:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())


### Locate the class folders

The code below automatically checks the expected directory structure and creates a dataframe containing every image path and its label.


In [ ]:
EXPECTED = [
    DATA_ROOT / "train",
    DATA_ROOT / "test",
    DATA_ROOT / "val"
]

for p in EXPECTED:
    print(p, "->", p.exists())

rows = []
for split in ["train", "test", "val"]:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        continue
    for label_name in ["NORMAL", "PNEUMONIA"]:
        class_dir = split_dir / label_name
        if class_dir.exists():
            for ext in ("*.jpeg", "*.jpg", "*.png"):
                for img_path in class_dir.glob(ext):
                    rows.append({
                        "path": str(img_path),
                        "label_name": label_name,
                        "label": 0 if label_name == "NORMAL" else 1,
                        "source_split": split
                    })

df = pd.DataFrame(rows)
print("Total images found:", len(df))
display(df.head())
print("\nClass distribution:")
display(df["label_name"].value_counts())


## 3. Display 5–10 random images from each class

This satisfies the assignment's visual exploration requirement. fileciteturn0file0L12-L15


In [ ]:
def show_random_images(dataframe, label_name, n=8):
    subset = dataframe[dataframe["label_name"] == label_name].sample(
        min(n, len(dataframe[dataframe["label_name"] == label_name])),
        random_state=SEED
    )
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.ravel()
    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = imread(row["path"], as_gray=True)
        ax.imshow(img, cmap="gray")
        ax.set_title(label_name)
        ax.axis("off")
    for ax in axes[len(subset):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_random_images(df, "NORMAL", 8)
show_random_images(df, "PNEUMONIA", 8)


In [ ]:
# Class balance
class_counts = df["label_name"].value_counts().reindex(["NORMAL", "PNEUMONIA"])
print(class_counts)

plt.figure(figsize=(7, 4))
class_counts.plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Class percentages:")
print((class_counts / class_counts.sum() * 100).round(2))


## 4. Task 2 – Preprocessing

The assignment requires:
- resize to a fixed dimension such as **128×128**
- grayscale conversion where required
- HOG
- LBP
- GLCM
- feature vectors
- 70% training / 15% validation / 15% testing

These steps are implemented below. fileciteturn0file0L16-L24


In [ ]:
def load_gray_image(path, size=IMG_SIZE):
    img = imread(path, as_gray=True)
    img = resize(img, size, anti_aliasing=True)
    return img.astype(np.float32)

# Example preprocessing
sample_path = df.sample(1, random_state=SEED)["path"].iloc[0]
sample_img = load_gray_image(sample_path)

plt.figure(figsize=(5, 5))
plt.imshow(sample_img, cmap="gray")
plt.title("Preprocessed 128×128 Grayscale Image")
plt.axis("off")
plt.show()


### HOG, LBP and GLCM feature extraction

For every image, a single combined feature vector is created:

**HOG + LBP histogram + GLCM texture statistics**

The resulting vectors can be used for classical-feature experiments and provide the requested feature extraction component of the assignment.


In [ ]:
def extract_features(path):
    img = load_gray_image(path)

    # HOG
    hog_features = hog(
        img,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    # LBP histogram
    radius = 2
    n_points = 8 * radius
    lbp = local_binary_pattern(img, n_points, radius, method="uniform")
    n_bins = n_points + 2
    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_bins + 1),
        range=(0, n_bins),
        density=True
    )

    # GLCM
    # Quantize grayscale values to 8 levels to keep computation lightweight.
    quantized = np.floor(img * 7).astype(np.uint8)
    glcm = graycomatrix(
        quantized,
        distances=[1, 2],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=8,
        symmetric=True,
        normed=True
    )

    glcm_props = []
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        glcm_props.extend(graycoprops(glcm, prop).ravel())

    return np.concatenate([hog_features, lbp_hist, np.asarray(glcm_props)])

# Test extraction on one image
test_features = extract_features(sample_path)
print("Combined feature-vector length:", len(test_features))


### Extract feature vectors for the full dataset

This can take several minutes because HOG/LBP/GLCM are computed image-by-image. The output is cached to `/content/chest_xray_features.npz` so it does not have to be recomputed in the same Colab session.


In [ ]:
FEATURE_CACHE = "/content/chest_xray_features.npz"

if os.path.exists(FEATURE_CACHE):
    cached = np.load(FEATURE_CACHE)
    X_features = cached["X"]
    y_features = cached["y"]
    print("Loaded cached features:", X_features.shape)
else:
    feature_list = []
    labels_list = []

    for i, row in enumerate(df.itertuples(index=False), start=1):
        try:
            feature_list.append(extract_features(row.path))
            labels_list.append(row.label)
        except Exception as e:
            print("Skipping:", row.path, "|", e)

        if i % 500 == 0:
            print(f"Processed {i}/{len(df)} images")

    X_features = np.vstack(feature_list).astype(np.float32)
    y_features = np.asarray(labels_list, dtype=np.int64)

    np.savez_compressed(FEATURE_CACHE, X=X_features, y=y_features)
    print("Saved:", FEATURE_CACHE)

print("Feature matrix:", X_features.shape)
print("Labels:", y_features.shape)


## 5. Train/Validation/Test split – 70/15/15

A stratified split is used so that both classes are represented proportionally in all three subsets. This directly follows the assignment requirement. fileciteturn0file0L23-L24


In [ ]:
# First split: 70% train, 30% temporary
X_train_feat, X_temp_feat, y_train_feat, y_temp_feat = train_test_split(
    X_features, y_features,
    test_size=0.30,
    stratify=y_features,
    random_state=SEED
)

# Second split: half of temporary -> 15% validation and 15% test
X_val_feat, X_test_feat, y_val_feat, y_test_feat = train_test_split(
    X_temp_feat, y_temp_feat,
    test_size=0.50,
    stratify=y_temp_feat,
    random_state=SEED
)

print("Train:", X_train_feat.shape, f"({len(X_train_feat)/len(X_features):.1%})")
print("Validation:", X_val_feat.shape, f"({len(X_val_feat)/len(X_features):.1%})")
print("Test:", X_test_feat.shape, f"({len(X_test_feat)/len(X_features):.1%})")


## 6. Optional feature-based sanity-check classifier

The assignment focuses on a CNN, but HOG/LBP/GLCM are feature-engineering requirements. This lightweight Logistic Regression model verifies that the extracted feature vectors can be used for classification.

The CNN remains the main model evaluated later.


In [ ]:
feature_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, class_weight="balanced"))
])

feature_clf.fit(X_train_feat, y_train_feat)
val_pred_feat = feature_clf.predict(X_val_feat)
val_prob_feat = feature_clf.predict_proba(X_val_feat)[:, 1]

print("Feature-based validation metrics")
print("Accuracy :", accuracy_score(y_val_feat, val_pred_feat))
print("Precision:", precision_score(y_val_feat, val_pred_feat, zero_division=0))
print("Recall   :", recall_score(y_val_feat, val_pred_feat, zero_division=0))
print("F1-score :", f1_score(y_val_feat, val_pred_feat, zero_division=0))
print("AUROC    :", roc_auc_score(y_val_feat, val_prob_feat))


# 7. CNN Model

The CNN uses 128×128 grayscale images. Data augmentation is used during training to reduce overfitting.

The model is intentionally moderate in size so it can run on a standard Google Colab GPU.


In [ ]:
IMAGE_DF = df.copy()

# 70/15/15 stratified split using file paths
train_df, temp_df = train_test_split(
    IMAGE_DF,
    test_size=0.30,
    stratify=IMAGE_DF["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("CNN split sizes:", len(train_df), len(val_df), len(test_df))

BATCH_SIZE = 32

def make_dataset(dataframe, training=False):
    paths = dataframe["path"].values
    labels = dataframe["label"].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_jpeg(image, channels=1)
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2000, seed=SEED)

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)


In [ ]:
def build_cnn(learning_rate=1e-3, dropout=0.30):
    model = keras.Sequential([
        layers.Input(shape=(128, 128, 1)),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),

        layers.Dropout(dropout),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model = build_cnn()
model.summary()


## 8. Hyperparameter tuning using cross-validation

A full CNN cross-validation over every image can be computationally expensive in a Colab environment. The following **3-fold stratified CV** uses a configurable subset and a small CNN for comparing learning-rate/dropout configurations.

After selecting the best configuration, the full training set is used for the final CNN.


In [ ]:
# Use a stratified subset for practical 3-fold CV.
# Increase CV_MAX_SAMPLES if you have more runtime/GPU available.
CV_MAX_SAMPLES = min(1800, len(train_df))

cv_df = train_df.sample(CV_MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

def small_cnn(lr=1e-3, dropout=0.30):
    model = keras.Sequential([
        layers.Input(shape=(128,128,1)),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )
    return model

param_grid = [
    {"lr": 1e-3, "dropout": 0.20},
    {"lr": 1e-3, "dropout": 0.40},
    {"lr": 3e-4, "dropout": 0.20},
    {"lr": 3e-4, "dropout": 0.40},
]

def paths_to_ds(dataframe, batch_size=32, training=False):
    ds = make_dataset(dataframe, training=training)
    return ds

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv_results = []

for params in param_grid:
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(cv_df["path"], cv_df["label"]), start=1):
        fold_train = cv_df.iloc[tr_idx]
        fold_val = cv_df.iloc[va_idx]

        m = small_cnn(params["lr"], params["dropout"])

        m.fit(
            paths_to_ds(fold_train, training=True),
            validation_data=paths_to_ds(fold_val),
            epochs=2,
            verbose=0
        )

        probs = m.predict(paths_to_ds(fold_val), verbose=0).ravel()
        preds = (probs >= 0.5).astype(int)
        auc = roc_auc_score(fold_val["label"], probs)
        fold_scores.append(auc)

        print(f"params={params}, fold={fold}, AUROC={auc:.4f}")

        del m
        tf.keras.backend.clear_session()

    cv_results.append({
        "learning_rate": params["lr"],
        "dropout": params["dropout"],
        "mean_cv_auroc": np.mean(fold_scores),
        "std_cv_auroc": np.std(fold_scores)
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "mean_cv_auroc", ascending=False
).reset_index(drop=True)

display(cv_results_df)

best_params = {
    "learning_rate": float(cv_results_df.loc[0, "learning_rate"]),
    "dropout": float(cv_results_df.loc[0, "dropout"])
}
print("Selected hyperparameters:", best_params)


## 9. Train the final CNN

The final model uses the best configuration found by cross-validation and trains on the complete 70% training split. Early stopping and learning-rate reduction are used to improve generalization.


In [ ]:
final_model = build_cnn(
    learning_rate=best_params["learning_rate"],
    dropout=best_params["dropout"]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = final_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)


## 10. Training curves


In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df["accuracy"], label="Train")
plt.plot(history_df["val_accuracy"], label="Validation")
plt.title("CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["loss"], label="Train")
plt.plot(history_df["val_loss"], label="Validation")
plt.title("CNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


## 11. Task 3 – Basic Evaluation

The assignment requires:
- Accuracy
- Precision
- Recall
- F1-score
- AUROC
- Confusion matrix
- explanation of misclassifications
- comparison and identification of the best performer

These are calculated on the held-out test set. fileciteturn0file0L25-L30


In [ ]:
# CNN predictions
test_probs = final_model.predict(test_ds, verbose=1).ravel()
test_true = test_df["label"].values
test_pred = (test_probs >= 0.5).astype(int)

cnn_metrics = {
    "Accuracy": accuracy_score(test_true, test_pred),
    "Precision": precision_score(test_true, test_pred, zero_division=0),
    "Recall": recall_score(test_true, test_pred, zero_division=0),
    "F1-score": f1_score(test_true, test_pred, zero_division=0),
    "AUROC": roc_auc_score(test_true, test_probs)
}

print("CNN Test Metrics")
for k, v in cnn_metrics.items():
    print(f"{k:10s}: {v:.4f}")

print("\nClassification Report")
print(classification_report(
    test_true,
    test_pred,
    target_names=["NORMAL", "PNEUMONIA"],
    zero_division=0
))


### Confusion Matrix


In [ ]:
cm = confusion_matrix(test_true, test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["NORMAL", "PNEUMONIA"],
    yticklabels=["NORMAL", "PNEUMONIA"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)


### Misclassification analysis

For this binary task:

- **False Positive:** a NORMAL X-ray is predicted as PNEUMONIA.
- **False Negative:** a PNEUMONIA X-ray is predicted as NORMAL.

False negatives are particularly important from a diagnostic-support perspective because an abnormal case may be missed. The confusion matrix above shows the number of each type of error.


In [ ]:
# Display examples of misclassified test images
test_display = test_df.copy()
test_display["prob_pneumonia"] = test_probs
test_display["pred"] = test_pred
test_display["correct"] = test_true == test_pred

errors = test_display[~test_display["correct"]].copy()

print("Total misclassified images:", len(errors))

if len(errors) > 0:
    sample_errors = errors.sample(min(12, len(errors)), random_state=SEED)

    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.ravel()

    for ax, (_, row) in zip(axes, sample_errors.iterrows()):
        img = imread(row["path"], as_gray=True)
        ax.imshow(img, cmap="gray")
        actual = "PNEUMONIA" if row["label"] == 1 else "NORMAL"
        predicted = "PNEUMONIA" if row["pred"] == 1 else "NORMAL"
        ax.set_title(
            f"Actual: {actual}\nPred: {predicted}\nP={row['prob_pneumonia']:.2f}"
        )
        ax.axis("off")

    for ax in axes[len(sample_errors):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples in this test run.")


## 12. AUROC Curve


In [ ]:
fpr, tpr, _ = roc_curve(test_true, test_probs)
auc_value = roc_auc_score(test_true, test_probs)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"CNN AUROC = {auc_value:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("CNN ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 13. Model Comparison

The feature-based HOG+LBP+GLCM classifier is included as a reference, while the CNN is the primary model requested by the assignment.


In [ ]:
# Feature-based model test metrics
feature_test_pred = feature_clf.predict(X_test_feat)
feature_test_prob = feature_clf.predict_proba(X_test_feat)[:, 1]

comparison = pd.DataFrame([
    {
        "Model": "HOG + LBP + GLCM + Logistic Regression",
        "Accuracy": accuracy_score(y_test_feat, feature_test_pred),
        "Precision": precision_score(y_test_feat, feature_test_pred, zero_division=0),
        "Recall": recall_score(y_test_feat, feature_test_pred, zero_division=0),
        "F1-score": f1_score(y_test_feat, feature_test_pred, zero_division=0),
        "AUROC": roc_auc_score(y_test_feat, feature_test_prob)
    },
    {
        "Model": "CNN",
        "Accuracy": cnn_metrics["Accuracy"],
        "Precision": cnn_metrics["Precision"],
        "Recall": cnn_metrics["Recall"],
        "F1-score": cnn_metrics["F1-score"],
        "AUROC": cnn_metrics["AUROC"]
    }
])

display(comparison.round(4))

best_model_name = comparison.sort_values("F1-score", ascending=False).iloc[0]["Model"]
print("Best performer based on F1-score:", best_model_name)


## 14. Final conclusion

Run the following cell after all experiments. It automatically creates a concise conclusion using the measured results rather than hard-coding values.


In [ ]:
best_row = comparison.sort_values("F1-score", ascending=False).iloc[0]

print("CONCLUSION")
print("-" * 70)
print(
    f"The CNN achieved an accuracy of {cnn_metrics['Accuracy']:.4f}, "
    f"precision of {cnn_metrics['Precision']:.4f}, recall of {cnn_metrics['Recall']:.4f}, "
    f"F1-score of {cnn_metrics['F1-score']:.4f}, and AUROC of {cnn_metrics['AUROC']:.4f} "
    f"on the held-out test set."
)
print(
    f"Based on F1-score, the best performer in the comparison was "
    f"'{best_model_name}'."
)
print(
    f"The confusion matrix contained {fn} false negatives and {fp} false positives. "
    "These errors represent cases where the model predicted the opposite class."
)
print(
    "HOG, LBP and GLCM were used to construct hand-crafted feature vectors, "
    "while the CNN learned image representations directly from the X-ray pixels."
)


## 15. Save the trained model

The trained CNN can be downloaded from Colab after running the next cell.


In [ ]:
MODEL_PATH = "/content/chest_xray_cnn.keras"
final_model.save(MODEL_PATH)
print("Saved model:", MODEL_PATH)

# Uncomment to download:
# from google.colab import files
# files.download(MODEL_PATH)


## Assignment checklist

| Requirement | Implemented |
|---|---|
| Download/load chest X-ray dataset | ✓ |
| 5–10 random images from each class | ✓ |
| Class balance | ✓ |
| Resize to 128×128 | ✓ |
| Grayscale preprocessing | ✓ |
| HOG | ✓ |
| LBP | ✓ |
| GLCM | ✓ |
| Feature vectors | ✓ |
| 70/15/15 split | ✓ |
| CNN model | ✓ |
| Cross-validation / hyperparameter tuning | ✓ |
| Accuracy | ✓ |
| Precision | ✓ |
| Recall | ✓ |
| F1-score | ✓ |
| AUROC | ✓ |
| Confusion matrix | ✓ |
| Misclassification examples | ✓ |
| Model comparison | ✓ |
